# Camada Silver — `matriculas`

**Disciplina:** Mineração de Dados — FATEC Matão-SP (Prof. Rodrigo Vidrich)  
**Projeto:** Trabalho Final (P2) — Arquitetura Medallion (`Bronze → Silver → Gold → Power BI`)  
**Integrantes:** Jerônimo Barbieri Junior, Luan Chierato Oliveira, Miguel Miranda Morandini, Victor Hugo Malipense Testi  
**Data:** 13/11/2026  

**Técnicas aplicadas:** Padronização de Tipos (`id_aluno`, `id_matricula`), Limpeza Textual (`faculdade`, `curso`, `semestre`), Padronização Categórica (`status`) e Remoção de Duplicatas.

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

for candidate in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
    if (candidate / "notebooks" / "utils").exists():
        if str(candidate / "notebooks") not in sys.path:
            sys.path.insert(0, str(candidate / "notebooks"))
        break

from utils.silver_utils import (
    anonymize_name_lgpd,
    detect_outliers_iqr,
    detect_outliers_zscore,
    fix_mojibake_text,
    get_silver_paths,
    load_bronze,
    normalize_id_aluno,
    parse_br_float,
    parse_date_col,
    save_silver_outputs,
)

ROOT_DIR, BRONZE_DIR, SILVER_DIR = get_silver_paths()

In [ ]:
df = load_bronze("matriculas", BRONZE_DIR)
rows_in = len(df)

df["id_matricula"] = normalize_id_aluno(df["id_matricula"])
df["id_aluno"] = normalize_id_aluno(df["id_aluno"])

for col in ["faculdade", "curso", "semestre", "status"]:
    df[col], _ = fix_mojibake_text(df[col])

df["faculdade"] = df["faculdade"].str.upper().astype("category")
df["curso"] = df["curso"].str.title().astype("category")
df["semestre"] = df["semestre"].str.strip().astype("category")
df["status"] = df["status"].str.title().astype("category")

dups_removed = int(df.duplicated(subset=["id_matricula"]).sum())
df = df.drop_duplicates(subset=["id_matricula"]).reset_index(drop=True)

log = {
    "dataset": "matriculas",
    "rows_in": rows_in,
    "rows_out": int(len(df)),
    "duplicates_removed": dups_removed,
    "status_counts": {str(k): int(v) for k, v in df["status"].value_counts().items()},
}
save_silver_outputs(df, log, "matriculas", SILVER_DIR)
print(json.dumps(log, ensure_ascii=False, indent=2))
df.head(5)